In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install -q -U transformers accelerate bitsandbytes peft trl datasets evaluate rouge_score bert_score sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 88.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 72.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 35.6 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conf

In [2]:
import gc
import json
import random
import re
from pathlib import Path

import torch
import evaluate

from datasets import Dataset
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
)

from trl import SFTTrainer, SFTConfig

In [3]:
# Config
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

HF_TOKEN = "YOUR_HF_TOKEN"

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
DATA_PATH = "/kaggle/input/datasets/hartzbyte/scientific-research-papers/dataset.json"

OUTPUT_DIR = Path("/kaggle/working/qlora_mistral_research")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ADAPTER_DIR = OUTPUT_DIR / "adapter"
EVAL_DIR = OUTPUT_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

MAX_TRAIN_SAMPLES = None      # set e.g. 1000 for debugging, else None for full train
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = 50         # keep the same held-out test subset for comparison
MAX_SEQ_LENGTH = 1024
MAX_NEW_TOKENS = 512

# Conservative Kaggle-friendly training config
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
NUM_TRAIN_EPOCHS = 2
LEARNING_RATE = 2e-4
LOGGING_STEPS = 10
SAVE_STRATEGY = "epoch"
EVAL_STRATEGY = "epoch"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56


In [4]:
# Hugging Face login
login("hf_...")

In [5]:
# Load raw dataset
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total raw samples:", len(data))
random.shuffle(data)

# same 80/10/10 split
n = len(data)
train_end = int(0.8 * n)
val_end = int(0.9 * n)

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

if MAX_TRAIN_SAMPLES:
    train_data = train_data[:MAX_TRAIN_SAMPLES]
if MAX_VAL_SAMPLES:
    val_data = val_data[:MAX_VAL_SAMPLES]
if MAX_TEST_SAMPLES:
    test_data = test_data[:MAX_TEST_SAMPLES]

print(f"Train: {len(train_data)}")
print(f"Val:   {len(val_data)}")
print(f"Test:  {len(test_data)}")

# save splits for reproducibility
for name, split in [
    ("train.json", train_data),
    ("val.json", val_data),
    ("test.json", test_data),
]:
    with open(OUTPUT_DIR / name, "w", encoding="utf-8") as f:
        json.dump(split, f, indent=2, ensure_ascii=False)

Total raw samples: 1500
Train: 1200
Val:   150
Test:  50


In [6]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [7]:
from collections import Counter

output_types = Counter(type(x["output"]).__name__ for x in data)
print(output_types)

Counter({'dict': 1493, 'str': 7})


In [8]:
def normalize_ws(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()

def coerce_output(output_obj):
    if isinstance(output_obj, dict):
        return output_obj, "dict"

    if isinstance(output_obj, str):
        raw = output_obj.strip()
        try:
            parsed = json.loads(raw)
            if isinstance(parsed, dict):
                return parsed, "json_string_dict"
        except Exception:
            pass
        return None, "plain_string"

    return None, "other"

def format_output_as_text(output_obj):
    parts = []
    for key, value in output_obj.items():
        parts.append(f"{key}:")
        if isinstance(value, list):
            for item in value:
                parts.append(f"- {normalize_ws(item)}")
        else:
            parts.append(normalize_ws(value))
    return "\n".join(parts).strip()

def build_user_prompt(instruction, abstract, expected_fields):
    field_lines = "\n".join([f"- {field}" for field in expected_fields])
    return (
        "You are analyzing a scientific research abstract.\n\n"
        f"Task:\n{instruction}\n\n"
        "Return the answer using only these fields:\n"
        f"{field_lines}\n\n"
        "Keep the answer faithful to the abstract.\n"
        "Do not invent unsupported claims.\n\n"
        f"Research Abstract:\n{abstract}"
    )

def make_chat_example(item, idx=None):
    structured_output, output_type = coerce_output(item["output"])

    if structured_output is None:
        return None, output_type

    expected_fields = list(structured_output.keys())
    user_prompt = build_user_prompt(item["instruction"], item["input"], expected_fields)
    assistant_text = format_output_as_text(structured_output)

    messages = [
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_text},
    ]

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {
        "text": full_text,
        "instruction": item["instruction"],
        "input": item["input"],
        "reference_text": assistant_text,
        "expected_fields": expected_fields,
        "output_type_detected": output_type,
        "original_index": idx,
    }, output_type

train_formatted = []
train_stats = Counter()

for i, item in enumerate(train_data):
    result, output_type = make_chat_example(item, idx=i)
    train_stats[output_type] += 1
    if result is not None:
        train_formatted.append(result)

val_formatted = []
val_stats = Counter()

for i, item in enumerate(val_data):
    result, output_type = make_chat_example(item, idx=i)
    val_stats[output_type] += 1
    if result is not None:
        val_formatted.append(result)

train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

print("Train output stats:", train_stats)
print("Val output stats:", val_stats)
print("Final train samples:", len(train_dataset))
print("Final val samples:", len(val_dataset))

print("\nSample formatted training example:\n")
print(train_dataset[0]["text"][:2000])

Train output stats: Counter({'dict': 1193, 'plain_string': 7})
Val output stats: Counter({'dict': 150})
Final train samples: 1193
Final val samples: 150

Sample formatted training example:

<s> [INST] You are analyzing a scientific research abstract.

Task:
Find contradictions in this research and identify its methodology.

Return the answer using only these fields:
- contradictions
- methods

Keep the answer faithful to the abstract.
Do not invent unsupported claims.

Research Abstract:
We introduce String Seed of Thought (SSoT), a novel prompting method for LLMs that improves Probabilistic Instruction Following (PIF). We define PIF as a task requiring an LLM to select its answer from a predefined set of options, each associated with a specific probability, such that the empirical distribution of the generated answers aligns with the target distribution when prompted multiple times. While LLMs excel at tasks with single, deterministic answers, they often fail at PIF, exhibiting biases

In [9]:
from collections import Counter

train_output_types = Counter(x["output_type_detected"] for x in train_formatted)
val_output_types = Counter(x["output_type_detected"] for x in val_formatted)

print("Train formatted output types:", train_output_types)
print("Val formatted output types:", val_output_types)

Train formatted output types: Counter({'dict': 1193})
Val formatted output types: Counter({'dict': 150})


In [10]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_use_double_quant=True,
)

torch.cuda.empty_cache()
gc.collect()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    torch_dtype=torch.float32,
)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.gradient_checkpointing_enable()

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [11]:
# Training arguments
sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR / "trainer_output"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=20,
    report_to="none",
    seed=42,
    max_grad_norm=0.0,
    packing=False,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
)

# SFT trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    args=sft_config,
    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/1193 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1193 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1193 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

In [12]:
# Train
train_result = trainer.train()

# Save adapter
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

with open(OUTPUT_DIR / "train_result.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "train_runtime": train_result.metrics.get("train_runtime"),
            "train_loss": train_result.metrics.get("train_loss"),
            "train_samples": len(train_dataset),
            "val_samples": len(val_dataset),
        },
        f,
        indent=2,
    )

print("\nSaved LoRA adapter to:", ADAPTER_DIR)

# free memory before evaluation reload
del trainer
del model
torch.cuda.empty_cache()
gc.collect()

Epoch,Training Loss,Validation Loss
1,1.252096,1.230647



Saved LoRA adapter to: /kaggle/working/qlora_mistral_research/adapter


46921

In [13]:
# Reload base model + adapter for evaluation
base_eval_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

ft_model = PeftModel.from_pretrained(base_eval_model, str(ADAPTER_DIR))
ft_model.eval()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_pro

In [20]:
# Evaluation helpers
import json
import evaluate
import torch

def normalize_ws(text: str) -> str:
    import re
    return re.sub(r"\s+", " ", str(text)).strip()

def coerce_output(output_obj):
    if isinstance(output_obj, dict):
        return output_obj, "dict"

    if isinstance(output_obj, str):
        raw = output_obj.strip()
        try:
            parsed = json.loads(raw)
            if isinstance(parsed, dict):
                return parsed, "json_string_dict"
        except Exception:
            pass
        return None, "plain_string"

    return None, "other"

def get_expected_fields(output_obj):
    structured_output, output_type = coerce_output(output_obj)
    if structured_output is None:
        return None, output_type
    return list(structured_output.keys()), output_type

def format_output_as_text(output_obj):
    parts = []
    for key, value in output_obj.items():
        parts.append(f"{key}:")
        if isinstance(value, list):
            for item in value:
                parts.append(f"- {normalize_ws(item)}")
        else:
            parts.append(normalize_ws(value))
    return "\n".join(parts).strip()

def flatten_output(output_obj):
    structured_output, output_type = coerce_output(output_obj)
    if structured_output is None:
        return None, output_type
    return format_output_as_text(structured_output), output_type

def build_eval_prompt(instruction, abstract, expected_fields):
    field_lines = "\n".join([f"- {field}" for field in expected_fields])
    messages = [
        {
            "role": "user",
            "content": (
                "You are analyzing a scientific research abstract.\n\n"
                f"Task:\n{instruction}\n\n"
                "Return the answer using only these fields:\n"
                f"{field_lines}\n\n"
                "Keep the answer faithful to the abstract.\n"
                "Do not invent unsupported claims.\n\n"
                f"Research Abstract:\n{abstract}"
            ),
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def schema_field_presence(text, expected_fields):
    text_lower = text.lower()
    return {field: f"{field.lower()}:" in text_lower for field in expected_fields}

def generate_response(model, prompt, max_new_tokens=512):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_len:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return decoded

In [21]:
# Evaluate the fine-tuned model on the held-out test set
predictions = []
references = []
records = []
skipped_eval = []

for idx, item in enumerate(test_data):
    expected_fields, expected_type = get_expected_fields(item["output"])
    reference_text, ref_type = flatten_output(item["output"])

    if expected_fields is None or reference_text is None:
        skipped_eval.append({
            "id": idx,
            "reason": ref_type,
            "instruction": item["instruction"],
        })
        continue

    prompt = build_eval_prompt(item["instruction"], item["input"], expected_fields)
    prediction_text = generate_response(ft_model, prompt, max_new_tokens=MAX_NEW_TOKENS)
    schema_check = schema_field_presence(prediction_text, expected_fields)

    predictions.append(prediction_text)
    references.append(reference_text)

    records.append({
        "id": idx,
        "instruction": item["instruction"],
        "input": item["input"],
        "expected_fields": expected_fields,
        "reference_text": reference_text,
        "prediction_text": prediction_text,
        "schema_check": schema_check,
        "reference_type": ref_type,
    })

    if (idx + 1) % 10 == 0 or (idx + 1) == len(test_data):
        print(f"Evaluated {idx + 1}/{len(test_data)}")

with open(EVAL_DIR / "finetuned_predictions.json", "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

with open(EVAL_DIR / "skipped_eval_samples.json", "w", encoding="utf-8") as f:
    json.dump(skipped_eval, f, indent=2, ensure_ascii=False)

print("Usable eval samples:", len(records))
print("Skipped eval samples:", len(skipped_eval))

Evaluated 10/50
Evaluated 20/50
Evaluated 30/50
Evaluated 40/50
Evaluated 50/50
Usable eval samples: 50
Skipped eval samples: 0


In [24]:
# Compute metrics
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

rouge_result = rouge.compute(predictions=predictions, references=references)
bertscore_result = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="en"
)

avg_bertscore_f1 = sum(bertscore_result["f1"]) / len(bertscore_result["f1"])

total_expected = 0
total_found = 0
exact_schema_match_count = 0

for rec in records:
    checks = rec["schema_check"]
    found_count = sum(1 for v in checks.values() if v)
    expected_count = len(checks)

    total_found += found_count
    total_expected += expected_count

    if found_count == expected_count:
        exact_schema_match_count += 1

schema_field_recall = total_found / total_expected if total_expected else 0.0
exact_schema_match = exact_schema_match_count / len(records) if records else 0.0

metrics = {
    "model_id": MODEL_ID,
    "adapter_path": str(ADAPTER_DIR),
    "num_test_samples_total": len(test_data),
    "num_test_samples_used": len(records),
    "num_test_samples_skipped": len(skipped_eval),
    "rouge": rouge_result,
    "avg_bertscore_f1": avg_bertscore_f1,
    "schema_field_recall": schema_field_recall,
    "exact_schema_match": exact_schema_match,
    "generation_config": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False
    },
    "train_config": {
        "epochs": NUM_TRAIN_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "max_seq_length": MAX_SEQ_LENGTH,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
    }
}

with open(EVAL_DIR / "finetuned_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

manual_review = []
for rec in records[:10]:
    manual_review.append({
        "instruction": rec["instruction"],
        "expected_fields": rec["expected_fields"],
        "reference_text": rec["reference_text"],
        "prediction_text": rec["prediction_text"],
    })

with open(EVAL_DIR / "manual_review_samples.json", "w", encoding="utf-8") as f:
    json.dump(manual_review, f, indent=2, ensure_ascii=False)

print("\n=== Fine-tuned Metrics ===")
print(json.dumps(metrics, indent=2))
print(f"\nArtifacts saved in: {OUTPUT_DIR}")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== Fine-tuned Metrics ===
{
  "model_id": "mistralai/Mistral-7B-Instruct-v0.2",
  "adapter_path": "/kaggle/working/qlora_mistral_research/adapter",
  "num_test_samples_total": 50,
  "num_test_samples_used": 50,
  "num_test_samples_skipped": 0,
  "rouge": {
    "rouge1": 0.6630158442347706,
    "rouge2": 0.47227248669420663,
    "rougeL": 0.5441058295425546,
    "rougeLsum": 0.6132428067051732
  },
  "avg_bertscore_f1": 0.9307381129264831,
  "schema_field_recall": 1.0,
  "exact_schema_match": 1.0,
  "generation_config": {
    "max_new_tokens": 512,
    "do_sample": false
  },
  "train_config": {
    "epochs": 2,
    "learning_rate": 0.0002,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "max_seq_length": 1024,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05
  }
}

Artifacts saved in: /kaggle/working/qlora_mistral_research
- adapter
- evaluation
- test.json
- train.json
- train_result.json
- trainer_output
- val.json


In [28]:
# Save a small manual review file
manual_review = []
for rec in records[:10]:
    manual_review.append({
        "instruction": rec["instruction"],
        "expected_fields": rec["expected_fields"],
        "reference_text": rec["reference_text"],
        "prediction_text": rec["prediction_text"],
    })

with open(EVAL_DIR / "manual_review_samples.json", "w", encoding="utf-8") as f:
    json.dump(manual_review, f, indent=2, ensure_ascii=False)

print(f"\nArtifacts saved in: {OUTPUT_DIR}")
print("Top-level files/folders:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)


Artifacts saved in: /kaggle/working/qlora_mistral_research
Top-level files/folders:
- adapter
- evaluation
- test.json
- train.json
- train_result.json
- trainer_output
- val.json
